**Note:** This notebook represents an early design exploration for DAG visualisation. The concepts herein (like 'disliked nodes', 'weak/strong' edges) are more complex than the current, simplified implementation in the main application.

In [1]:
import matplotlib.pyplot as plt\nimport networkx as nx\nimport random\nfrom matplotlib import colormaps as cm\nfrom matplotlib.colors import LinearSegmentedColormap\nimport numpy as np\nimport matplotlib.patches as mpatches\n\n# Create directed graph\nG = nx.DiGraph()\nG.add_nodes_from(range(1, 10))\n\n# Add in different node types\ndisliked_nodes = [3]\ntransaction_nodes = [2, 5, 9]\ndata_nodes = [1, 4, 6, 7, 8]\n\n# Create list of edges\n# (parent, child, strength)\nedges = [\n    (2, 1, 'strong'),  \n    (3, 1, 'strong'),\n    (4, 1, 'strong'),  \n    (4, 2, 'strong'),  \n    (5, 3, 'strong'),  \n    (5, 4, 'strong'),  \n    (6, 3, 'strong'),  \n    (6, 4, 'strong'),  \n    (7, 4, 'strong'),   \n    (7, 6, 'weak'),   \n    (8, 2, 'strong'),    \n    (8, 4, 'strong'),\n    (9, 7, 'strong'),   \n]\n\n# Add edges and nodes to graph\nfor child, parent, ref_type in edges:\n    G.add_edge(child, parent, reference=ref_type)\n\nnode_type = {}\nnode_weak = {}\n\n# Initialise all nodes as strong originally\nfor node in G.nodes:\n    node_weak[node] = False\n\n# Check if node is actually weak\n# Weak: has a strong connection to a disliked parent node, or if parent node is weak\nfor node in G.nodes:\n    incoming = list(G.in_edges(node, data=True))\n    for child, parent, data in incoming:\n        if ((parent in disliked_nodes and data['reference'] == 'strong') or node_weak[parent]) and not node_weak[child]:\n            node_weak[child] = True\n    node_type[node] = 'weak' if node_weak[node] else 'strong'\n\n# Add node positions in 2D\n# Evenly distributed in x, for visualisation, and randomly in Y to avoid overlap\nrandom.seed(42)\nlayer_map = {\n    1: 0,\n    2: 1, 3: 1,\n    4: 2, 5: 2,\n    6: 3, 7: 4,\n    8: 4, 9: 4\n}\npos = {}\nfor node, layer in layer_map.items():\n    x = layer * 2.5\n    y = random.uniform(-1, 1)\n    pos[node] = (x, y)\nnx.set_node_attributes(G, pos, 'pos')\n

In [ ]:
# Draw plot\nfig, ax = plt.subplots(figsize=(15, 8))\n\n# Node degree (number of connections)\ndegrees = dict(G.degree())\nnorm = plt.Normalize(vmin=min(degrees.values()), vmax=max(degrees.values()))\n\n# Truncate colour map to avoid bright yellow\ndef truncate_colormap(cmap, minval=0.0, maxval=1.0, n=256):\n    new_cmap = LinearSegmentedColormap.from_list(\n        f'trunc({cmap.name},{minval:.2f},{maxval:.2f})',\n        cmap(np.linspace(minval, maxval, n))\n    )\n    return new_cmap\n\ncmap = truncate_colormap(cm.get_cmap('viridis'), 0.0, 0.7)\n\n# Collect strong and weak nodes separately\nstrong_nodes = [n for n in G.nodes() if node_type[n] == 'strong']\nweak_nodes = [n for n in G.nodes() if node_type[n] == 'weak']\n\n# Build node colours and borders\nnode_colors = []\nnode_borders = []\n\n# Set node borders and colours\n# Colours relate to node degree\n# Borders relate to node type (disliked, transaction or other)\nfor node in G.nodes():\n    degree_norm = norm(degrees[node])\n    node_colors.append(cmap(degree_norm))\n    \n    if node in disliked_nodes:\n        node_borders.append('firebrick')\n    elif node in transaction_nodes:\n        node_borders.append('royalblue')\n    else:\n        node_borders.append('black')\n\n# Draw strong nodes\nnx.draw_networkx_nodes(\n    G,\n    pos,\n    nodelist=strong_nodes,\n    node_color=[node_colors[n-1] for n in strong_nodes],\n    edgecolors=[node_borders[n-1] for n in strong_nodes],\n    node_size=800,\n    linewidths=3,\n    ax=ax\n)\n\n# Draw weak nodes\nnx.draw_networkx_nodes(\n    G,\n    pos,\n    nodelist=weak_nodes,\n    node_color=[node_colors[n-1] for n in weak_nodes],\n    edgecolors=[node_borders[n-1] for n in weak_nodes],\n    node_size=800,\n    linewidths=0.1,\n    ax=ax\n)\n\n# Draw edges\nstrong_edges = [(u, v) for u, v, d in G.edges(data=True) if d['reference'] == 'strong']\nweak_edges = [(u, v) for u, v, d in G.edges(data=True) if d['reference'] == 'weak']\n\nnx.draw_networkx_edges(\n    G,\n    pos,\n    edgelist=strong_edges,\n    edge_color='#888',\n    arrows=True,\n    arrowsize=15,\n    width=1,\n    connectionstyle=\"arc3,rad=0.0\",\n    ax=ax,\n    min_source_margin=13,\n    min_target_margin=13\n)\n\nnx.draw_networkx_edges(\n    G,\n    pos,\n    edgelist=weak_edges,\n    edge_color='firebrick',\n    style='dotted',\n    arrows=True,\n    arrowsize=15,\n    width=1,\n    connectionstyle=\"arc3,rad=0.0\",\n    ax=ax,\n    min_source_margin=12,\n    min_target_margin=12\n)\n\n# Node labels\n# Shows the node's ID in white\nlabels = {node: f\"{node}\" for node in G.nodes()}\nnx.draw_networkx_labels(G, pos, labels, font_size=9, font_color='white', ax=ax)\n\n# Add Colourbar\nsm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)\nsm.set_array([])\ncbar = fig.colorbar(sm, ax=ax, shrink=0.7)\ncbar.set_label('Number of Connections (Degree)')\n\n# Add Legend\nlegend_elements = [\n    mpatches.Patch(facecolor='#eee', edgecolor='gray', label='Data Node'),\n    mpatches.Patch(facecolor='#eee', edgecolor='royalblue', label='Transaction Node'),\n    mpatches.Patch(facecolor='#eee', edgecolor='firebrick', label='Disliked Node'),\n    plt.Line2D([0], [0], color='black', lw=3, linestyle='solid', label='Strong Node'),\n    plt.Line2D([0], [0], color='black', lw=0.1, linestyle='solid', label='Weak Node'),\n    plt.Line2D([0], [0], color='#888', lw=2, label='Strong Edge'),\n    plt.Line2D([0], [0], color='firebrick', lw=2, linestyle='dotted', label='Weak Edge')\n]\nax.legend(handles=legend_elements, loc='lower left', fontsize='small')\n\n# Show plot\nax.set_title(\"Directed Acyclic Graph\", fontsize=14)\nax.axis('off')\n\nplt.tight_layout()\nplt.show()